<a href="https://colab.research.google.com/github/producer-valentyn/sql-analytics-portfolio/blob/main/notebooks/ab_testing_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 A/B Testing Statistical Significance Analysis

## 🎯 Business Goal
The goal of this project is to evaluate whether product changes have a statistically significant impact on user behavior.
## 🧠 What I Did

- Calculated conversion rates for control and test groups
- Implemented Z-test for proportions using Python
- Computed p-values and statistical significance
- Built a scalable solution using loops (no hardcoding)
- Prepared clean dataset for Tableau visualization

This project analyzes A/B test results using statistical methods.

We calculate conversion rates and perform a Z-test for proportions to determine whether differences between control and test groups are statistically significant.

Metrics analyzed:
- add_payment_info / session
- add_shipping_info / session
- begin_checkout / session
- new_accounts / session

## 🔬 Statistical Method

We use a Z-test for proportions to compare conversion rates between control and test groups.

- Null hypothesis (H0): no difference between groups
- Alternative hypothesis (H1): there is a difference

Significance level (alpha): 0.05

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


%cd /content/drive/MyDrive/Mate/Google\ Colab

Mounted at /content/drive
/content/drive/MyDrive/Mate/Google Colab


In [ ]:
# --- Load raw A/B test data ---
# This dataset contains raw counts for control and test groups

df = pd.read_csv("ab_test_results.csv")

df.head()

,test_number,metric,numerator_control,denominator_control,numerator_test,denominator_test
0,1,add_payment_info,1988,45362,2229,45193
1,1,add_shipping_info,3034,45362,3221,45193
2,1,begin_checkout,3784,45362,4021,45193
3,1,new_accounts,3823,45362,3681,45193
4,2,add_payment_info,2344,50637,2409,50244


In [ ]:
df['test_number'].unique()

array([1, 2, 3, 4])

In [ ]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   test_number          16 non-null     int64 
 1   metric               16 non-null     object
 2   numerator_control    16 non-null     int64 
 3   denominator_control  16 non-null     int64 
 4   numerator_test       16 non-null     int64 
 5   denominator_test     16 non-null     int64 
dtypes: int64(5), object(1)
memory usage: 900.0+ bytes


,test_number,numerator_control,denominator_control,numerator_test,denominator_test
count,16.000000,16.00000,16.000000,16.000000,16.000000
mean,2.500000,5099.18750,67781.250000,5065.625000,67754.250000
std,1.154701,2876.15714,24181.556921,2738.572764,24334.314947
min,1.000000,1988.00000,45362.000000,2229.000000,45193.000000
25%,1.750000,3587.25000,49318.250000,3578.250000,48981.250000
50%,2.500000,3994.00000,60342.000000,4102.500000,60341.500000
75%,3.250000,5437.50000,78805.000000,5346.500000,79114.500000
max,4.000000,12555.00000,105079.000000,12267.000000,105141.000000


In [ ]:
# --- Calculate A/B test metrics ---
# Conversion rates, metric change, z-test and p-value
metric_names = {
    'add_payment_info': 'add_payment_info / session',
    'add_shipping_info': 'add_shipping_info / session',
    'begin_checkout': 'begin_checkout / session',
    'new_accounts': 'new_accounts / session'
}

df['metric_full'] = df['metric'].map(metric_names)

In [ ]:
df['conversion_rate_control'] = df['numerator_control'] / df['denominator_control']
df['conversion_rate_test'] = df['numerator_test'] / df['denominator_test']

In [ ]:
df['metric_change'] = (
    (df['conversion_rate_test'] - df['conversion_rate_control'])
    / df['conversion_rate_control']
) * 100

In [ ]:
p1 = df['conversion_rate_control']
p2 = df['conversion_rate_test']
n1 = df['denominator_control']
n2 = df['denominator_test']

p_pool = (df['numerator_control'] + df['numerator_test']) / (n1 + n2)

se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))

df['z_stat'] = (p2 - p1) / se

In [ ]:
df['p_value'] = 2 * (1 - norm.cdf(np.abs(df['z_stat'])))

df['significant'] = df['p_value'] < 0.05

In [ ]:
# --- Final dataset for Tableau ---
df_final = df.copy()

df_final['numerator_event'] = df_final['numerator_control']
df_final['denominator_event'] = df_final['denominator_control']

df_final = df_final[[
    'test_number',
    'metric',

    'numerator_event',
    'denominator_event',

    'numerator_control',
    'denominator_control',

    'numerator_test',
    'denominator_test',

    'conversion_rate_control',
    'conversion_rate_test',

    'metric_change',
    'z_stat',
    'p_value',
    'significant'
]]

# rename
df_final = df_final.rename(columns={
    'numerator_control': 'numerator_count_control',
    'denominator_control': 'denominator_count_control',

    'numerator_test': 'numerator_count_test',
    'denominator_test': 'denominator_count_test',

    'metric_change': 'metric_change'
})

df_final.head()

,test_number,metric,numerator_event,denominator_event,numerator_count_control,denominator_count_control,numerator_count_test,denominator_count_test,conversion_rate_control,conversion_rate_test,metric_change,z_stat,p_value,significant
0,1,add_payment_info / session,1988,45362,1988,45362,2229,45193,0.043825,0.049322,12.542021,3.924884,0.000087,True
1,1,add_shipping_info / session,3034,45362,3034,45362,3221,45193,0.066884,0.071272,6.560481,2.603571,0.009226,True
2,1,begin_checkout / session,3784,45362,3784,45362,4021,45193,0.083418,0.088974,6.660587,2.978783,0.002894,True
3,1,new_accounts / session,3823,45362,3823,45362,3681,45193,0.084278,0.081451,-3.354299,-1.542883,0.122859,False
4,2,add_payment_info / session,2344,50637,2344,50637,2409,50244,0.046290,0.047946,3.576911,1.240994,0.214608,False


In [ ]:
df_final = df_final.rename(columns={

    'numerator_control': 'numerator_count_control',
    'denominator_control': 'denominator_count_control',

    'numerator_test': 'numerator_count_test',
    'denominator_test': 'denominator_count_test',

    'metric_change': 'metric_change'
})

In [ ]:
# --- Save final dataset ---
# This CSV contains all calculated metrics for A/B test analysis

output_path = "/content/drive/MyDrive/Mate/Google Colab/ab_test_results_final.csv"

df_final.to_csv(output_path, index=False)

## 📌 Key Insights

- Some product changes significantly improved conversion rates
- Not all metrics showed statistically significant differences
- This indicates that certain funnel steps are more sensitive to changes
- Results can be used to prioritize product improvements

### Additional Insight

To enhance the analysis, metrics were ranked by absolute change.
This helps identify which user actions were most impacted by the experiment.

This approach allows prioritizing product improvements based on impact.

## 🔗 Project Links

- 📊 Tableau Dashboard:  
https://public.tableau.com/app/profile/valentyn.fedyk/viz/ab_testing_project/Dashboard4

- 📁 CSV Results File:  
https://drive.google.com/file/d/1eNCk-h9TYQgQDLQC7Q8D57HNZx3cTMVI/view?usp=sharing